# 17. Stacking Adversarial Attack

**Tujuan:** Simulasi Saliency Map & FGSM evasion attack terhadap Stacking Ensemble.
Apakah ensemble lebih tahan terhadap adversarial dibanding single XGBoost?

**Input:** `stacking_baseline_15.pkl`, `adversarial_results_05.pkl` (single model comparison)

**Output:** `stacking_adversarial_17.pkl`, PNG visualisasi vulnerability curves

In [ ]:
import sys
!{sys.executable} -m pip install lightgbm catboost scikit-learn xgboost matplotlib -q

import numpy as np
import pandas as pd
import pickle
import os
import time
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
DATA_DIR = '../data/'
EPSILONS = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2]
MAX_SAMPLES = 50000
CACHE_FILE = os.path.join(DATA_DIR, 'stacking_adversarial_17.pkl')

# === CHECK CACHE ===
if os.path.exists(CACHE_FILE):
    print(f'Cache found: {CACHE_FILE} — loading (skip computation)...')
    with open(CACHE_FILE, 'rb') as f:
        cached = pickle.load(f)
    SKIP_TRAINING = True
else:
    print('No cache. Will run full adversarial analysis.')
    SKIP_TRAINING = False

print('Libraries loaded.')

## 1. Load Stacking Model & Data

In [ ]:
with open(os.path.join(DATA_DIR, 'stacking_baseline_15.pkl'), 'rb') as f:
    stack_data = pickle.load(f)

X_train = stack_data['X_train']
X_test = stack_data['X_test']
y_train = stack_data['y_train']
y_test = stack_data['y_test']
base_models = stack_data['base_models']
meta_learner = stack_data['meta_learner']
scaler_meta = stack_data['scaler_meta']
n_classes = stack_data['n_classes']
feature_names = stack_data['feature_names']

# Use Top-10 features for adversarial analysis
xgb_model = base_models['XGBoost']
importances = xgb_model.feature_importances_
top10_idx = np.argsort(importances)[::-1][:10]

# Subset to Top-10 for saliency computation
X_test_10 = X_test[:, top10_idx]
X_train_10 = X_train[:, top10_idx]

print(f'Test set: {X_test.shape[0]} samples')
print(f'Top-10 features selected for adversarial analysis')

## 2. Stacking Prediction Function

In [ ]:
def stacking_predict(X_input, base_models, meta_learner, scaler_meta, n_classes):
    """
    Full stacking inference: base learners → meta-features → meta-learner → prediction
    """
    meta_features = np.zeros((X_input.shape[0], n_classes * len(base_models)))
    for idx, (name, model) in enumerate(base_models.items()):
        col_s = idx * n_classes
        col_e = (idx + 1) * n_classes
        meta_features[:, col_s:col_e] = model.predict_proba(X_input)
    
    meta_scaled = scaler_meta.transform(meta_features)
    return meta_learner.predict(meta_scaled)

def stacking_predict_proba(X_input, base_models, meta_learner, scaler_meta, n_classes):
    """Return probability predictions from stacking."""
    meta_features = np.zeros((X_input.shape[0], n_classes * len(base_models)))
    for idx, (name, model) in enumerate(base_models.items()):
        col_s = idx * n_classes
        col_e = (idx + 1) * n_classes
        meta_features[:, col_s:col_e] = model.predict_proba(X_input)
    
    meta_scaled = scaler_meta.transform(meta_features)
    return meta_learner.predict_proba(meta_scaled)

# Verify baseline prediction
y_pred_clean = stacking_predict(X_test, base_models, meta_learner, scaler_meta, n_classes)
mcc_clean = matthews_corrcoef(y_test, y_pred_clean)
f1_clean = f1_score(y_test, y_pred_clean, average='weighted')
print(f'Stacking Baseline (clean): MCC={mcc_clean:.4f}, F1={f1_clean*100:.2f}%')

## 3. Compute Saliency Map for Stacking

In [ ]:
def compute_loss_stacking(X_input, y_true, base_models, meta_learner, scaler_meta, n_classes):
    """Compute cross-entropy loss for stacking ensemble."""
    probs = stacking_predict_proba(X_input, base_models, meta_learner, scaler_meta, n_classes)
    probs = np.clip(probs, 1e-10, 1.0)
    n_samples = X_input.shape[0]
    loss = -np.mean(np.log(probs[np.arange(n_samples), y_true.astype(int)]))
    return loss

def compute_saliency_stacking(X, y_true, base_models, meta_learner, scaler_meta, n_classes, h=0.01):
    """Compute saliency map via finite differences for stacking."""
    n_samples, n_features = X.shape
    saliency = np.zeros_like(X)
    
    for i in range(n_features):
        X_plus = X.copy()
        X_plus[:, i] += h
        loss_plus = compute_loss_stacking(X_plus, y_true, base_models, meta_learner, scaler_meta, n_classes)
        
        X_minus = X.copy()
        X_minus[:, i] -= h
        loss_minus = compute_loss_stacking(X_minus, y_true, base_models, meta_learner, scaler_meta, n_classes)
        
        saliency[:, i] = (loss_plus - loss_minus) / (2 * h)
    
    return saliency

# Compute saliency on subset
print(f'Computing saliency map on {MAX_SAMPLES} samples...')
X_sal = X_test[:MAX_SAMPLES]
y_sal = y_test[:MAX_SAMPLES]

saliency = compute_saliency_stacking(
    X_sal, y_sal, base_models, meta_learner, scaler_meta, n_classes, h=0.01
)
mean_saliency = np.mean(np.abs(saliency), axis=0)
print(f'Done. Most sensitive feature: {feature_names[np.argmax(mean_saliency)]} (saliency={mean_saliency.max():.4f})')

## 4. FGSM Attack at Multiple Epsilon

In [ ]:
def generate_adversarial(X, saliency, epsilon):
    """Generate adversarial samples using FGSM."""
    perturbation = epsilon * np.sign(saliency)
    return X + perturbation

vulnerability_results = []

print('='*60)
print('  VULNERABILITY ANALYSIS: Stacking vs Epsilon')
print('='*60)

for eps in EPSILONS:
    if eps == 0:
        X_adv = X_sal.copy()
    else:
        X_adv = generate_adversarial(X_sal, saliency, eps)
    
    y_pred_adv = stacking_predict(X_adv, base_models, meta_learner, scaler_meta, n_classes)
    mcc_adv = matthews_corrcoef(y_sal, y_pred_adv)
    f1_adv = f1_score(y_sal, y_pred_adv, average='weighted')
    
    vulnerability_results.append({
        'epsilon': eps,
        'mcc': mcc_adv,
        'f1_score': f1_adv
    })
    print(f'  ε={eps:.3f} | MCC={mcc_adv:.4f} | F1={f1_adv*100:.2f}%')

print('='*60)

## 5. Comparison: Stacking vs Single XGBoost Vulnerability

In [ ]:
# Load single model vulnerability from notebook 05
with open(os.path.join(DATA_DIR, 'adversarial_results_05.pkl'), 'rb') as f:
    single_adv = pickle.load(f)

single_vuln = single_adv.get('vulnerability_results', [])

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

eps_stack = [r['epsilon'] for r in vulnerability_results]
mcc_stack = [r['mcc'] for r in vulnerability_results]
f1_stack = [r['f1_score'] for r in vulnerability_results]

# MCC comparison
axes[0].plot(eps_stack, mcc_stack, 'b-o', linewidth=2, label='Stacking Ensemble')
if single_vuln:
    eps_single = [r.get('epsilon', 0) for r in single_vuln]
    mcc_single = [r.get('mcc', 0) for r in single_vuln]
    axes[0].plot(eps_single, mcc_single, 'r--s', linewidth=2, label='Single XGBoost')
axes[0].set_xlabel('Epsilon (ε)')
axes[0].set_ylabel('MCC')
axes[0].set_title('MCC Degradation: Stacking vs Single Model')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 comparison
axes[1].plot(eps_stack, [f*100 for f in f1_stack], 'b-o', linewidth=2, label='Stacking Ensemble')
if single_vuln:
    f1_single = [r.get('f1_score', 0) for r in single_vuln]
    axes[1].plot(eps_single, [f*100 for f in f1_single], 'r--s', linewidth=2, label='Single XGBoost')
axes[1].set_xlabel('Epsilon (ε)')
axes[1].set_ylabel('F1-Score (%)')
axes[1].set_title('F1 Degradation: Stacking vs Single Model')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'stacking_vulnerability_comparison.png'), bbox_inches='tight')
plt.show()
print('Saved: stacking_vulnerability_comparison.png')

## 6. Save Results

In [ ]:
output = {
    'saliency': saliency,
    'mean_saliency': mean_saliency,
    'vulnerability_results': vulnerability_results,
    'X_test_subset': X_sal,
    'y_test_subset': y_sal,
    'epsilons': EPSILONS,
    'baseline_mcc_clean': mcc_clean,
    'baseline_f1_clean': f1_clean,
    'feature_names': feature_names
}

with open(os.path.join(DATA_DIR, 'stacking_adversarial_17.pkl'), 'wb') as f:
    pickle.dump(output, f)

print('Saved: stacking_adversarial_17.pkl')
print('\nNotebook 17 selesai. Lanjut ke 18 (Adversarial Training pada Stacking).')